# 06. LSTM Sentiment Analysis (Improved)
Train an advanced bidirectional LSTM model with contextual attention on sentiment data for 10 epochs.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
import re
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


d:\financial statment analyzer\financialvenv\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cpu


In [2]:
# Load data - using the enhanced dataset generated recently
df = pd.read_csv('../data/processed/merged_sentiment_data.csv')

# Drop missing values if any
df = df.dropna(subset=['text', 'sentiment'])

# Encode sentiment labels
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_mapping)

print(f"Total samples: {len(df)}")
print("Sentiment distribution:")
print(df['sentiment'].value_counts())

# Train-test split (90% training, 10% testing)
train_df, test_df = train_test_split(df, test_size=0.10, random_state=42, stratify=df['label'])
print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

# For class weighting:
counts = train_df['label'].value_counts().sort_index()
class_counts = torch.tensor([counts.get(0, 0), counts.get(1, 0), counts.get(2, 0)], dtype=torch.float)
class_weights = (1.0 / class_counts)
class_weights = class_weights / class_weights.sum()
print(f"Computed Class Weights: {class_weights}")


Total samples: 14297
Sentiment distribution:
sentiment
neutral     6279
positive    4714
negative    3304
Name: count, dtype: int64
Training samples: 12867
Testing samples: 1430
Computed Class Weights: tensor([0.4490, 0.2363, 0.3148])


In [3]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)
    return text.split()

# Build vocabulary
vocab = Counter()
for text in train_df['text']:
    vocab.update(tokenize(text))

# Remove rare words
min_freq = 2
vocab = {word: count for word, count in vocab.items() if count >= min_freq}

# Create word to index mapping
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word in vocab:
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)
print(f"Vocabulary size: {vocab_size}")

def encode_sentence(text, max_len=50):
    tokens = tokenize(text)
    encoded = [word2idx.get(word, word2idx['<UNK>']) for word in tokens]
    # mask = 1 for real tokens, 0 for pad
    mask = [1]*len(encoded)
    
    if len(encoded) < max_len:
        padding_len = max_len - len(encoded)
        encoded += [word2idx['<PAD>']] * padding_len
        mask += [0] * padding_len
    else:
        encoded = encoded[:max_len]
        mask = mask[:max_len]
    return encoded, mask

MAX_LEN = 50
encoded_train = [encode_sentence(text, MAX_LEN) for text in train_df['text']]
X_train = np.array([e[0] for e in encoded_train])
M_train = np.array([e[1] for e in encoded_train])
y_train = np.array(train_df['label'])

encoded_test = [encode_sentence(text, MAX_LEN) for text in test_df['text']]
X_test = np.array([e[0] for e in encoded_test])
M_test = np.array([e[1] for e in encoded_test])
y_test = np.array(test_df['label'])


Vocabulary size: 7898


In [4]:
class SentimentDataset(Dataset):
    def __init__(self, X, mask, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.mask = torch.tensor(mask, dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]

batch_size = 64
train_dataset = SentimentDataset(X_train, M_train, y_train)
test_dataset = SentimentDataset(X_test, M_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [5]:
class ImprovedLSTMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        output_dim,
        n_layers,
        bidirectional,
        dropout,
        pad_idx=0
    ):
        super().__init__()
        
        # Save pad_idx properly
        self.pad_idx = pad_idx

        # Separate embedding dropout (avoids zeroing same token every time)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(dropout * 0.5)  # lighter dropout on embeddings

        # Layer Normalization after embedding for stable training
        self.embed_layer_norm = nn.LayerNorm(embedding_dim)

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,  # dropout only between layers
            batch_first=True
        )

        self.direction_factor = 2 if bidirectional else 1

        # Attention mechanism — focuses on most sentiment-relevant tokens
        self.attention = nn.Linear(hidden_dim * self.direction_factor, 1)

        # Deeper classifier head with BatchNorm and residual signal
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * self.direction_factor, hidden_dim), # Just using context now, no last_hidden concat
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, output_dim)
        )

        self.dropout = nn.Dropout(dropout)
        self._init_weights()

    def _init_weights(self):
        """Xavier/Orthogonal initialization for stable gradient flow."""
        nn.init.xavier_uniform_(self.embedding.weight)
        # Use proper padding index reference!
        self.embedding.weight.data[self.embedding.padding_idx].fill_(0)

        for name, param in self.lstm.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)  # orthogonal for recurrent weights
            elif 'bias' in name:
                param.data.fill_(0)
                # Set forget gate bias to 1 — helps remember long-term dependencies
                n = param.size(0)
                param.data[n // 4: n // 2].fill_(1)

        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def attention_pooling(self, lstm_output, mask=None):
        """
        Soft attention over all timesteps.
        lstm_output: [batch, seq_len, hidden * directions]
        mask:        [batch, seq_len] bool — True for real tokens, False for padding
        """
        # [batch, seq_len, 1] → [batch, seq_len]
        attn_scores = self.attention(lstm_output).squeeze(-1)

        if mask is not None:
            # Mask must be properly evaluated as bool
            attn_scores = attn_scores.masked_fill(mask == False, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=1)          # [batch, seq_len]
        context = torch.bmm(attn_weights.unsqueeze(1), lstm_output).squeeze(1)  # [batch, hidden*dir]
        return context, attn_weights

    def forward(self, text, mask=None):
        """
        text: [batch, seq_len]
        mask: [batch, seq_len] bool — True for real tokens
        """
        # --- Embedding ---
        embedded = self.embedding(text)                        # [batch, seq_len, emb_dim]
        embedded = self.embed_layer_norm(embedded)
        
        # Zero out padding tokens prior to feeding to LSTM to combat LayerNorm bias shifting pad values!
        if mask is not None:
            embedded = embedded * mask.unsqueeze(-1).float()
            
        embedded = self.embedding_dropout(embedded)

        # --- LSTM ---
        # Note: best practice would be pack_padded_sequence, but masking prior bounds the problem.
        lstm_out, (hidden, _) = self.lstm(embedded)
        lstm_out = self.dropout(lstm_out)                      # [batch, seq_len, hid*dir]

        # --- Attention pooling ---
        context, attn_weights = self.attention_pooling(lstm_out, mask)  # [batch, hid*dir]

        # Pass safety-checked attention context to classifier
        return self.classifier(context)                       # [batch, output_dim]

# Hyperparameters
EMBEDDING_DIM = 200
HIDDEN_DIM = 256
OUTPUT_DIM = 3
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.4

model = ImprovedLSTMClassifier(
    vocab_size    = vocab_size,
    embedding_dim = EMBEDDING_DIM,
    hidden_dim    = HIDDEN_DIM,
    output_dim    = OUTPUT_DIM,
    n_layers      = N_LAYERS,
    bidirectional = BIDIRECTIONAL,
    dropout       = DROPOUT,
    pad_idx       = word2idx['<PAD>']
)
model = model.to(device)
print(model)


ImprovedLSTMClassifier(
  (embedding): Embedding(7898, 200, padding_idx=0)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (embed_layer_norm): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
  (lstm): LSTM(200, 256, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (attention): Linear(in_features=512, out_features=1, bias=True)
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=3, bias=True)
  )
  (dropout): Dropout(p=0.4, inplace=False)
)


In [6]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

EPOCHS = 10
# Cosine annealing scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

def calculate_accuracy(preds, y):
    top_pred = preds.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc

def train_step(model, batch_text, batch_mask, batch_labels):
    model.train()
    optimizer.zero_grad()
    predictions = model(batch_text, mask=batch_mask)
    loss = criterion(predictions, batch_labels)
    acc = calculate_accuracy(predictions, batch_labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item(), acc.item()


In [7]:
train_losses = []
train_accs = []
best_loss = float('inf')

for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_acc = 0
    
    for batch_X, batch_M, batch_y in train_loader:
        batch_X, batch_M, batch_y = batch_X.to(device), batch_M.to(device), batch_y.to(device)
        
        loss, acc = train_step(model, batch_X, batch_M, batch_y)
        epoch_loss += loss
        epoch_acc += acc
        
    scheduler.step()
    
    avg_loss = epoch_loss / len(train_loader)
    avg_acc = epoch_acc / len(train_loader)
    
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)


In [8]:
model.eval()
test_loss = 0
test_acc = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_M, batch_y in test_loader:
        batch_X, batch_M, batch_y = batch_X.to(device), batch_M.to(device), batch_y.to(device)
        predictions = model(batch_X, mask=batch_M)
        
        loss = criterion(predictions, batch_y)
        acc = calculate_accuracy(predictions, batch_y)
        
        test_loss += loss.item()
        test_acc += acc.item()
        
        all_preds.extend(predictions.argmax(1).cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
avg_test_acc = test_acc / len(test_loader)

print(f'Test Loss: {avg_test_loss:.3f} | Test Acc: {avg_test_acc*100:.2f}%')
print("\nClassification Report:")
target_names = ['Negative', 'Neutral', 'Positive']
print(classification_report(all_labels, all_preds, target_names=target_names))


Test Loss: 1.502 | Test Acc: 78.89%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.80      0.80       330
     Neutral       0.78      0.81      0.80       628
    Positive       0.78      0.75      0.76       472

    accuracy                           0.79      1430
   macro avg       0.79      0.79      0.79      1430
weighted avg       0.79      0.79      0.79      1430



In [9]:
torch.save(model.state_dict(), '../data/processed/improved_lstm_sentiment_model.pt')
print("Improved model saved to ../data/processed/improved_lstm_sentiment_model.pt")


Improved model saved to ../data/processed/improved_lstm_sentiment_model.pt


In [13]:
i = 1
print("epoch no, train loss, train accuracy")
for train_loss, train_acc in zip(train_losses, train_accs): 
    print(f"{i} {train_loss:.3f} {train_acc:.3f}")
    i += 1


epoch no, train loss, train accuracy
1 0.769 0.679
2 0.383 0.857
3 0.220 0.921
4 0.127 0.954
5 0.077 0.976
6 0.042 0.988
7 0.024 0.993
8 0.015 0.996
9 0.011 0.997
10 0.008 0.998
